##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [2]:
# 1. Imports libraries
import os
import cv2
import numpy as np
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, GlobalAveragePooling2D , BatchNormalization

In [3]:
# 2. dataset path and classes
DATASET_PATH= r"C:\Users\Roros\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\action"
CLASSES = ["biking", "horse_riding", "tennis_swing"]

image_height, image_width = 64, 64
max_images_per_class = 8000

In [4]:
# 3. Load Video and extract frames
def frames_extraction(video_path):
    # Empty List declared to store video frames
    frames_list = []
    
    # Reading the Video File Using the VideoCapture
    video_reader = cv2.VideoCapture(video_path)

    # Iterating through Video Frames
    while True:

        # Reading a frame from the video file 
        success, frame = video_reader.read() 

        # If Video frame was not successfully read then break the loop
        if not success:
            break

        # Resize the Frame to fixed Dimensions
        resized_frame = cv2.resize(frame, (image_height, image_width))
        
        # Normalize the resized frame by dividing it with 255 so that each pixel value then lies between 0 and 1
        normalized_frame = resized_frame / 255
        
        # Appending the normalized frame into the frames list
        frames_list.append(normalized_frame)
    
    # Closing the VideoCapture object and releasing all resources. 
    video_reader.release()

    # returning the frames list 
    return frames_list

In [ ]:
# 4. Build Dataset
def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(CLASSES):
        class_path = os.path.join(DATASET_PATH, class_name)

        temp_features = []

       
        for root, dirs, files in os.walk(class_path):
            for file_name in files:
                if file_name.lower().endswith((".avi", ".mp4", ".mov", ".mpg")):
                    video_path = os.path.join(root, file_name)

                    frames = frames_extraction(video_path)
                    temp_features.extend(frames)

        print(f"{class_name}: {len(temp_features)} frames")

        if len(temp_features) == 0:
            print(f" WARNING: No videos found in {class_name}")
            continue

        sample_size = min(len(temp_features), max_images_per_class)

        selected_frames = random.sample(temp_features, sample_size)

        features.extend(selected_frames)
        labels.extend([class_index] * sample_size)

    return np.array(features), np.array(labels)

In [14]:
features, labels = create_dataset()

biking: 25272 frames
horse_riding: 34523 frames
tennis_swing: 24096 frames


In [15]:
one_hot_encoded_labels = to_categorical(labels)

In [16]:
# 6. split Train/Test
features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels, test_size=0.2, random_state=42)

In [ ]:
# 7. Build Model 
def create_model():

    # We will use a Sequential model for model construction
    model = Sequential()

    # Defining The Model Architecture
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu', input_shape = (image_height, image_width, 3)))
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2, 2)))
    model.add(GlobalAveragePooling2D())
    model.add(Dense(256, activation = 'relu'))
    model.add(BatchNormalization())
    model.add(Dense(len(CLASSES), activation = 'softmax'))

    # Printing the models summary
    model.summary()

    return model

# Calling the create_model method
model = create_model()

print("Model Created Successfully!")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,411 (224.26 KB)

 Trainable params: 56,771 (221.76 KB)

 Non-trainable params: 640 (2.50 KB)

Model Created Successfully!


In [18]:
# 8. Compile & train
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 15, mode = 'min', restore_best_weights = True)

# Adding loss, optimizer and metrics values to the model.
model.compile(loss = 'categorical_crossentropy', optimizer = 'Adam', metrics = ["accuracy"])

# Start Training
model_training_history = model.fit(x = features_train, y = labels_train, epochs = 15, batch_size = 4 , shuffle = True, validation_split = 0.2, callbacks = [early_stopping_callback])


Epoch 1/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 107s 27ms/step - accuracy: 0.5759 - loss: 0.9220 - val_accuracy: 0.6737 - val_loss: 1.1146
Epoch 2/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 101s 26ms/step - accuracy: 0.7744 - loss: 0.5365 - val_accuracy: 0.8570 - val_loss: 0.4009
Epoch 3/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 97s 25ms/step - accuracy: 0.8283 - loss: 0.4338 - val_accuracy: 0.8393 - val_loss: 0.6578
Epoch 4/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 97s 25ms/step - accuracy: 0.8628 - loss: 0.3643 - val_accuracy: 0.8458 - val_loss: 0.4136
Epoch 5/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 99s 26ms/step - accuracy: 0.8759 - loss: 0.3364 - val_accuracy: 0.9260 - val_loss: 0.2034
Epoch 6/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 97s 25ms/step - accuracy: 0.8928 - loss: 0.2870 - val_accuracy: 0.9602 - val_loss: 0.1135
Epoch 7/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 98s 25ms/step - accuracy: 0.9118 - loss: 0.2493 - val_accuracy: 0.9547 - val_loss: 0.1334
Epoch 8/15
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 101s 26ms/step - accuracy: 0.911

In [19]:
# 10. Evaluate
loss, acc = model.evaluate(features_test, labels_test)
print("Test Accuracy:", acc)


150/150 ━━━━━━━━━━━━━━━━━━━━ 9s 56ms/step - accuracy: 0.9880 - loss: 0.0370
Test Accuracy: 0.9870833158493042


In [20]:
# 11. Save Model 
student_name = "Raneem Alqahtani" 
save_path = "Raneem_Alqahtani_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as Raneem_Alqahtani_ucf11_model.h5


In [ ]:
from collections import deque
def predict_on_live_video(video_file_path, output_file_path, window_size):

    # Initialize a Deque Object with a fixed size which will be used to implement moving/rolling average functionality.
    predicted_labels_probabilities_deque = deque(maxlen = window_size)

    # Reading the Video File using the VideoCapture Object
    video_reader = cv2.VideoCapture(video_file_path)

    # Getting the width and height of the video 
    original_video_width = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_video_height = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Writing the Overlayed Video Files Using the VideoWriter Object
    video_writer = cv2.VideoWriter(output_file_path, cv2.VideoWriter_fourcc('M', 'P', '4', 'V'), 24, (original_video_width, original_video_height))

    while True: 

        # Reading The Frame
        status, frame = video_reader.read() 

        if not status:
            break

        # Resize the Frame to fixed Dimensions
        resized_frame = cv2.resize(frame, (image_height, image_width))
        
        # Normalize the resized frame by dividing it with 255 so that each pixel value then lies between 0 and 1
        normalized_frame = resized_frame / 255

        # Passing the Image Normalized Frame to the model and receiving Predicted Probabilities.
        predicted_labels_probabilities = model.predict(np.expand_dims(normalized_frame, axis = 0),verbose=0)[0]

        # Appending predicted label probabilities to the deque object
        predicted_labels_probabilities_deque.append(predicted_labels_probabilities)

        # Assuring that the Deque is completely filled before starting the averaging process
        if len(predicted_labels_probabilities_deque) == window_size:

            # Converting Predicted Labels Probabilities Deque into Numpy array
            predicted_labels_probabilities_np = np.array(predicted_labels_probabilities_deque)

            # Calculating Average of Predicted Labels Probabilities Column Wise 
            predicted_labels_probabilities_averaged = predicted_labels_probabilities_np.mean(axis = 0)

            # Converting the predicted probabilities into labels by returning the index of the maximum value.
            predicted_label = np.argmax(predicted_labels_probabilities_averaged)

            # Accessing The Class Name using predicted label.
            predicted_class_name = CLASSES[predicted_label]
          
            # Overlaying Class Name Text Ontop of the Frame
            cv2.putText(frame, predicted_class_name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Writing The Frame
        video_writer.write(frame)
        
    # Closing the VideoCapture and VideoWriter objects and releasing all resources held by them. 
    video_reader.release()
    video_writer.release()

In [22]:
test_folder = r"C:\Users\Roros\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\testing"
window_size = 25
output_directory=r"C:\Users\Roros\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\result"
for file in os.listdir(test_folder):
    if file.endswith((".mp4", ".avi", ".mov")):

        input_video = os.path.join(test_folder, file)
        output_video = os.path.join(output_directory, f"output_{file}")

        predict_on_live_video(input_video, output_video, window_size)